# Root Cause Analysis

1. Aggregate transaction-level data.
To make the analysis more meaningful, I introduced controlled perturbations to simulate noticeable growth patterns.

2. Select a target merchant and benchmark key business metrics (e.g., transaction amount, transaction count) against peer merchants within the same category.

In [1]:
import pandas as pd
import numpy as np
import etl
from itertools import combinations

from config import DIMENSION_COLS


In [2]:
# Load transaction data


df_comparison = etl.load_data()

/Users/yanxinye/github/competency-intelligence/ci/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
df_comparison.head()

,generation,state,gender,category,merchant,amt_ly,amt_ty
0,Boomer,AL,F,entertainment,fraud_Abbott-Rogahn,4.110,82.600000
1,Boomer,AL,F,entertainment,fraud_Abshire PLC,206.715,29.081219
2,Boomer,AL,F,entertainment,fraud_Bauch-Blanda,0.000,0.000000
3,Boomer,AL,F,entertainment,fraud_Beier LLC,0.000,0.000000
4,Boomer,AL,F,entertainment,fraud_Bins-Tillman,77.965,983.120000


Given a target merchant, create a peer set under the same category, and compare the business metrics against it. 

In [3]:
TARGET_MERCHANTS = "fraud_Kilback LLC" 

df_combined = etl.preprocess(df_comparison, TARGET_MERCHANTS)

Total amount in LY: 90116.45826107437
Total amount difference: 108492.6530555585
Total growth: 120.39%
Target industry of fraud_Kilback LLC: ['food_dining', 'grocery_pos']
Average growth of target: 120.39%
Average growth of peers: 94.52%
Average growth difference between target and peers: 25.87%


In [4]:
df_combined.head()

,generation,state,gender,category,merchant_target,amt_ly,amt_ty,amt_diff,amt_growth_ctc,amt_ty_peer,amt_ly_peer,amt_diff_peer,amt_growth_ctc_peer,merchant_peer,amt_growth_ctc_diff
0,Boomer,AL,F,food_dining,fraud_Kilback LLC,323.040000,776.92,453.880000,0.005037,294.358221,111.020952,183.337270,0.001763,ex-fraud_Kilback LLC,0.003273
1,Boomer,AL,F,grocery_pos,fraud_Kilback LLC,213.750000,0.00,-213.750000,-0.002372,385.485086,280.136657,105.348429,0.001013,ex-fraud_Kilback LLC,-0.003385
2,Boomer,AL,M,food_dining,fraud_Kilback LLC,191.370000,0.00,-191.370000,-0.002124,116.033160,101.086665,14.946496,0.000144,ex-fraud_Kilback LLC,-0.002267
3,Boomer,AL,M,grocery_pos,fraud_Kilback LLC,68.841556,974.84,905.998444,0.010054,341.726490,238.168357,103.558134,0.000996,ex-fraud_Kilback LLC,0.009058
4,Boomer,AR,F,food_dining,fraud_Kilback LLC,8.715000,0.00,-8.715000,-0.000097,129.949393,58.063871,71.885521,0.000691,ex-fraud_Kilback LLC,-0.000788


## Root Cause Analysis

In [5]:
df_combined.sort_values(by="amt_growth_ctc_diff", ascending=False).head()

,generation,state,gender,category,merchant_target,amt_ly,amt_ty,amt_diff,amt_growth_ctc,amt_ty_peer,amt_ly_peer,amt_diff_peer,amt_growth_ctc_peer,merchant_peer,amt_growth_ctc_diff
545,Millennial,TX,F,grocery_pos,fraud_Kilback LLC,1192.993940,8094.64,6901.646060,0.076586,3895.925881,1786.410407,2109.515474,0.020291,ex-fraud_Kilback LLC,0.056295
269,Gen X,NY,F,grocery_pos,fraud_Kilback LLC,173.480000,6040.20,5866.720000,0.065102,2157.218700,937.360481,1219.858219,0.011734,ex-fraud_Kilback LLC,0.053368
167,Gen X,CA,F,grocery_pos,fraud_Kilback LLC,888.458749,5724.92,4836.461251,0.053669,1583.183763,964.689093,618.494670,0.005949,ex-fraud_Kilback LLC,0.047720
531,Millennial,PA,F,grocery_pos,fraud_Kilback LLC,851.775000,4752.72,3900.945000,0.043288,1966.006811,934.293435,1031.713376,0.009924,ex-fraud_Kilback LLC,0.033364
439,Millennial,GA,M,grocery_pos,fraud_Kilback LLC,149.175000,3326.92,3177.745000,0.035263,387.123329,134.481102,252.642227,0.002430,ex-fraud_Kilback LLC,0.032833


In [6]:
target_col = "amt_growth_ctc_diff"

def grpby_dim_val(df, dim, target_col):
    kv = {}
    for i in df[[target_col, dim]].groupby(dim).sum().reset_index().to_dict("records"):

        k = "{} = {}".format(dim, i[dim])
        v = i[target_col]
        kv.update({k: v})
    return kv


res = {}
for dim in DIMENSION_COLS:
    res.update(grpby_dim_val(df_combined, dim, target_col))

res
               

{'generation = Boomer': -0.026154580370409283,
 'generation = Gen X': 0.1409785058528681,
 'generation = Gen Z': 0.04217049081909939,
 'generation = Millennial': 0.11142497033787593,
 'generation = Unknown': -0.009698757938148592,
 'state = AK': 0.0008035831663092299,
 'state = AL': 0.02110529744852842,
 'state = AR': 0.008043891042281457,
 'state = AZ': 0.02778962678680367,
 'state = CA': 0.07940780993199537,
 'state = CO': -0.005777093415928829,
 'state = CT': -0.006696088490714657,
 'state = DC': -0.0106194591967056,
 'state = DE': 0.0032924468429108944,
 'state = FL': 0.004663515186054068,
 'state = GA': 0.02719966835136314,
 'state = HI': 0.0019122430431895987,
 'state = IA': -0.00020631917465406893,
 'state = ID': -0.01002556609733837,
 'state = IL': 0.022018568496416552,
 'state = IN': 0.036191059476547556,
 'state = KS': -0.02688552851552631,
 'state = KY': 0.0033788604376908723,
 'state = LA': 0.007019722405038718,
 'state = MA': -0.0027200736183304085,
 'state = MD': 0.014564

In [9]:

target_col = "amt_growth_ctc_diff"

def grpby_dim_val(df, dims, target_col):
    kv = {}

    # make dims always a list
    if isinstance(dims, str):
        dims = [dims]

    grouped = (
        df[dims + [target_col]]
        .groupby(dims, dropna=False)[target_col]
        .sum()
        .reset_index()
    )

    for row in grouped.to_dict("records"):
        k_parts = [f"{dim} = {row[dim]}" for dim in dims]
        k = ", ".join(k_parts)
        v = row[target_col]
        kv[k] = v

    return kv


res = {}

# single dims
for dim in DIMENSION_COLS:
    res.update(grpby_dim_val(df_combined, dim, target_col))

# pairs of dims
for pair in combinations(DIMENSION_COLS, 2):
    res.update(grpby_dim_val(df_combined, list(pair), target_col))

In [10]:
res

{'generation = Boomer': -0.026154580370409283,
 'generation = Gen X': 0.1409785058528681,
 'generation = Gen Z': 0.04217049081909939,
 'generation = Millennial': 0.11142497033787593,
 'generation = Unknown': -0.009698757938148592,
 'state = AK': 0.0008035831663092299,
 'state = AL': 0.02110529744852842,
 'state = AR': 0.008043891042281457,
 'state = AZ': 0.02778962678680367,
 'state = CA': 0.07940780993199537,
 'state = CO': -0.005777093415928829,
 'state = CT': -0.006696088490714657,
 'state = DC': -0.0106194591967056,
 'state = DE': 0.0032924468429108944,
 'state = FL': 0.004663515186054068,
 'state = GA': 0.02719966835136314,
 'state = HI': 0.0019122430431895987,
 'state = IA': -0.00020631917465406893,
 'state = ID': -0.01002556609733837,
 'state = IL': 0.022018568496416552,
 'state = IN': 0.036191059476547556,
 'state = KS': -0.02688552851552631,
 'state = KY': 0.0033788604376908723,
 'state = LA': 0.007019722405038718,
 'state = MA': -0.0027200736183304085,
 'state = MD': 0.014564